# Asking questions about images using a VLM

VLM = Visual Language Model

#Setup


In [ ]:
#@title ▶ Install the required tools and setup the environment

!pip install -q transformers datasets num2words

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()   # only show errors

from datasets import logging as ds_logging
ds_logging.set_verbosity_error()   # only show errors


from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
#@title ▶ Download model

#@markdown Check use_big_model for slower, but more powerful, processing
use_big_model = False # @param {type:"boolean"}

from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

if use_big_model:
  model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
else:
  model_path = "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    # _attn_implementation="flash_attention_2"
).to("cuda")

In [ ]:
#@title ▶ Load a dataset (from [Huggingface](https://huggingface.co/datasets))  * Skip if you are using your own images

from datasets import load_dataset

dataset = load_dataset("valhalla/emoji-dataset")

images = dataset["train"]["image"]

## If you want to use your own images

Check 'use_own_dataset' and run, it will connect to drive, then in the next cell input the folder path where you have the images and run it.

The best way to get the path is to find the folder in the file explorer on the left, hover over it, click on the three dots at the right and select "copy path". Then paste it in the text box.

In [ ]:
#@markdown Select if you want to use your own images
use_own_dataset = False # @param {type:"boolean"}

if use_own_dataset:
  from google.colab import drive
  drive.mount('/content/drive')

In [ ]:
#@markdown Specify the path to your images on Drive

images_path = '/content/drive/MyDrive/tmp/BAU' # @param {type:"string"}

if use_own_dataset:
  from pathlib import Path

  image_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
  image_folder = Path(images_path)

  images = [img for img in image_folder.glob('*')
            if img.suffix.lower() in image_extensions]

  print(f"Found {len(images)} images in {images_path}")

#Processing

In [ ]:
#@title ▶ Process the images with the defined prompt

#@markdown Prompt
prompt = "What are emotions that the image evokes? Be brief, make a list and list a maximum of 3 emotions." # @param {type:"string"}

#@markdown Amount of images to process (set to a big number to process all the images)
amount = 5 # @param {type:"number"}

def process_image(image, prompt):
  import pathlib

  if type(image) != pathlib.PosixPath:
    image.save("/tmp/tmp.png", format="PNG")
    image = "/tmp/tmp.png"
  else:
    image = str(image)

  messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "path": image},
            {"type": "text", "text": prompt},
        ]
    },
  ]

  inputs = processor.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
  ).to(model.device, dtype=torch.bfloat16)

  generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=64)
  generated_texts = processor.batch_decode(
      generated_ids,
      skip_special_tokens=True,
  )

  return generated_texts[0]


from tqdm.notebook import tqdm

emotions = []
for idx in tqdm(range(min(amount, len(images)))):
  output = process_image(images[idx], prompt)

  image_emotions = output.split("\nAssistant:")[1]

  emotions.append(image_emotions)

In [ ]:
#@title Print the output

import pathlib

image_names = []

for idx, emotion in enumerate(emotions):
  if type(images[idx]) == pathlib.PosixPath:
    image_name = images[idx].name
  else:
    image_name = dataset["train"]["text"][idx]

  image_names.append(image_name)

image_names_max_len = max([len(name) for name in image_names])
image_names_max_len += 4

for emotion, image_name in zip(emotions, image_names):
  print(f"{image_name:<{image_names_max_len}}{emotion}")

# Credits

Taller Estampa https://tallerestampa.com / https://github.com/estampa
